# Differential Equations — Session 43
## Section 9.4: Higher-Order Equations and Systems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to reduce higher-order equations and coupled higher-order systems to first-order vector systems, apply vector Euler and RK4 methods, compare numerical methods on oscillatory problems, monitor conserved quantities, recognize numerical phase error, and use adaptive solvers for nonlinear systems.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Second-order IVP as a system |
| 18–38 min | Vector Euler method |
| 38–58 min | Vector RK4 method |
| 58–72 min | Oscillations, energy, and phase error |
| 72–84 min | Coupled systems |
| 84–90 min | Adaptive solver extension and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=8, suppress=True)
def euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n): ys[k+1]=ys[k]+h*f(xs[k],ys[k])
    return xs,ys
def improved_euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        p=ys[k]+h*f(xs[k],ys[k])
        ys[k+1]=ys[k]+0.5*h*(f(xs[k],ys[k])+f(xs[k+1],p))
    return xs,ys
def midpoint_rk2(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        ys[k+1]=ys[k]+h*k2
    return xs,ys
def rk4(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        k3=f(xs[k]+h/2,ys[k]+h*k2/2); k4=f(xs[k]+h,ys[k]+h*k3)
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return xs,ys
def rk4_system(f,t0,y0,h,n):
    ts=t0+h*np.arange(n+1); y0=np.asarray(y0,float)
    ys=np.zeros((n+1,len(y0))); ys[0]=y0
    for k in range(n):
        k1=np.asarray(f(ts[k],ys[k]))
        k2=np.asarray(f(ts[k]+h/2,ys[k]+h*k1/2))
        k3=np.asarray(f(ts[k]+h/2,ys[k]+h*k2/2))
        k4=np.asarray(f(ts[k]+h,ys[k]+h*k3))
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return ts,ys
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Principle 9.4-A — Second-order reduction

For

$$
y''=f(x,y,y'),
$$

let

$$
u=y'.
$$

Then

$$
\begin{aligned}
y'&=u,\\
u'&=f(x,y,u).
\end{aligned}
$$

### Principle 9.4-B — $m$th-order reduction

Introduce one state variable for each derivative up to order $m-1$.

### Definition 9.4-C — Vector Euler method

For

$$
\mathbf Y'=\mathbf F(t,\mathbf Y),
$$

$$
\mathbf Y_{n+1}
=
\mathbf Y_n+h\mathbf F(t_n,\mathbf Y_n).
$$

### Method 9.4-D — Vector RK4

The scalar RK4 formula applies with each $k_i$ replaced by a vector.

### Principle 9.4-E — Component coupling

Every RK stage must use a consistent temporary vector for all components. Updating components one at a time changes the method.

### Definition 9.4-F — Phase error

For oscillatory problems, a numerical method may preserve approximate amplitude while gradually shifting the oscillation phase.

### Principle 9.4-G — Invariant monitoring

When the exact system conserves a quantity such as energy, numerical drift in that quantity is a diagnostic of method quality.

### Classroom Checkpoint — Simultaneous Vector Stages

In vector RK4, why must every component use the same temporary stage vector?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. A second-order example

Consider

$$
y''+xy'+y=0,
\qquad
y(0)=1,
\qquad
y'(0)=2.
$$

Set $u=y'$:

$$
y'=u,
\qquad
u'=-xu-y.
$$

In [ ]:
def rhs_example(x,z):
    y,u=z
    return np.array([u,-x*u-y])

h=0.1
ts,ys=rk4_system(rhs_example,0,[1,2],h,20)
plt.plot(ts,ys[:,0],label="y")
plt.plot(ts,ys[:,1],label="y'")
plt.legend(); plt.title("RK4 for a second-order IVP"); plt.show()

## 2. Euler versus RK4 for a harmonic oscillator

For

$$
y''+y=0,
\qquad
y(0)=1,\quad y'(0)=0,
$$

the exact energy

$$
E=\frac12(y^2+u^2)
$$

is constant.

In [ ]:
def vector_euler(f,t0,y0,h,n):
    ts=t0+h*np.arange(n+1)
    y0=np.asarray(y0,float)
    ys=np.zeros((n+1,len(y0))); ys[0]=y0
    for k in range(n):
        ys[k+1]=ys[k]+h*np.asarray(f(ts[k],ys[k]))
    return ts,ys

osc=lambda t,z:np.array([z[1],-z[0]])
h=0.1; n=400
te,ye=vector_euler(osc,0,[1,0],h,n)
tr,yr=rk4_system(osc,0,[1,0],h,n)

plt.plot(ye[:,0],ye[:,1],label="Euler")
plt.plot(yr[:,0],yr[:,1],label="RK4")
plt.axis("equal"); plt.legend(); plt.title("Numerical phase portraits"); plt.show()

Ee=0.5*np.sum(ye**2,axis=1)
Er=0.5*np.sum(yr**2,axis=1)
plt.plot(te,Ee,label="Euler energy")
plt.plot(tr,Er,label="RK4 energy")
plt.legend(); plt.title("Energy drift"); plt.show()

In [ ]:
def oscillator_method_explorer(h=0.2, method="Euler"):
    osc=lambda t,z:np.array([z[1],-z[0]])
    final=40
    n=int(round(final/h)); h=final/n
    if method=="Euler":
        t,y=vector_euler(osc,0,[1,0],h,n)
    else:
        t,y=rk4_system(osc,0,[1,0],h,n)

    exact=np.cos(t)
    plt.plot(t,y[:,0],label=method)
    plt.plot(t,exact,linestyle="--",label="exact")
    plt.xlim(max(0,final-15),final)
    plt.legend(); plt.title(fr"$h={h:.3f}$"); plt.show()
    print("final state error:",np.linalg.norm(y[-1]-[np.cos(final),-np.sin(final)]))

if WIDGETS_AVAILABLE:
    interact(
        oscillator_method_explorer,
        h=FloatSlider(min=0.02,max=0.5,step=0.02,value=0.2),
        method=Dropdown(options=["Euler","RK4"],value="Euler")
    )
else:
    oscillator_method_explorer()

## 3. A coupled first-order system

For

$$
x'=2x+4y,
\qquad
y'=-x+6y,
$$

apply RK4 to the vector field as a whole.

In [ ]:
A=np.array([[2,4],[-1,6]],float)
system=lambda t,z:A@z
for h in [0.2,0.1]:
    t,z=rk4_system(system,0,[-1,6],h,int(round(0.6/h)))
    print("h =",h,"approximation at 0.6 =",z[-1])

exact=expm(A*0.6)@np.array([-1,6],float)
print("matrix-exponential exact value:",exact)

## 4. A coupled second-order mechanical system

Two masses lead to four first-order equations. Numerical methods do not require a closed-form modal solution.

In [ ]:
def coupled_masses(t,z):
    x1,x2,v1,v2=z
    k1,k2,m1,m2=4.0,2.0,1.0,1.5
    a1=(-(k1+k2)*x1+k2*x2)/m1
    a2=(k2*x1-k2*x2)/m2
    return np.array([v1,v2,a1,a2])

t,z=rk4_system(coupled_masses,0,[1,0,0,0],0.02,2500)
plt.plot(t,z[:,0],label=r"$x_1$")
plt.plot(t,z[:,1],label=r"$x_2$")
plt.legend(); plt.title("Coupled-mass oscillations"); plt.show()

## 5. Nonlinear system extension

The Lorenz equations illustrate why numerical solvers are essential when no elementary solution is available. Nearby initial conditions may separate strongly.

In [ ]:
def lorenz(t,z,sigma=10,rho=28,beta=8/3):
    x,y,zv=z
    return [sigma*(y-x),x*(rho-zv)-y,x*y-beta*zv]

grid=np.linspace(0,30,8000)
sol1=solve_ivp(lorenz,(0,30),[1,1,1],t_eval=grid,rtol=1e-9,atol=1e-11)
sol2=solve_ivp(lorenz,(0,30),[1.000001,1,1],t_eval=grid,rtol=1e-9,atol=1e-11)

distance=np.linalg.norm(sol1.y-sol2.y,axis=0)
plt.semilogy(grid,distance)
plt.xlabel("t"); plt.ylabel("trajectory separation")
plt.title("Sensitivity to initial conditions")
plt.show()

## 6. Adaptive versus fixed-step output

Adaptive solvers choose internal step sizes according to an error tolerance. The requested output grid does not reveal all internal steps.

In [ ]:
def vdp(t,z,mu=8):
    y,u=z
    return [u,mu*(1-y**2)*u-y]

for tol in [1e-3,1e-6,1e-9]:
    sol=solve_ivp(vdp,(0,30),[2,0],rtol=tol,atol=tol*1e-2)
    print("tolerance",tol,"accepted internal steps",len(sol.t))

## Classroom Checkpoint — Exit Check

Rewrite

$$
y'''+2y''-y'+4y=0
$$

as a first-order system.

> Pause here. Let students commit to an answer before running the next cell.